# 02 — Retrieval Strategies: Dense vs Sparse vs Hybrid

**Vai trò:** Pipeline Engineer · **Task:** S3-PE-08 (Yêu cầu 9.3)

Notebook này đào sâu vào bước **Retrieve** của RAG — bọc bởi `Retriever` (S3-PE-02), vốn gọi `ChromaVectorStore.similarity_search()` (S3-DE-01). Ta so sánh ba chiến lược truy xuất trên cùng một tập chunk:

- **Dense** — so khớp theo *ý nghĩa* qua embedding vector (cách `Retriever`/`ChromaVectorStore` đang dùng).
- **Sparse** — so khớp theo *từ khoá* xuất hiện chung giữa câu hỏi và đoạn văn bản (baseline kiểu BM25/TF-IDF, cài đặt đơn giản ngay trong notebook để minh hoạ).
- **Hybrid** — kết hợp cả hai điểm số (trung bình có trọng số) rồi xếp hạng lại.

Mục tiêu là quan sát trực quan: dense "hiểu ý nghĩa" tốt hơn khi câu hỏi diễn đạt khác từ ngữ với tài liệu, còn sparse nhạy với từ khoá trùng khớp chính xác — hybrid cố gắng tận dụng cả hai.

In [1]:
import re
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import DocumentLoader
from src.data.chunker import TextChunker
from src.embeddings.embedding_model import OllamaEmbeddingModel
from src.embeddings.vector_store import ChromaVectorStore
from src.retrieval.retriever import Retriever
from src.models import ChunkStrategy

print(f"Project root: {PROJECT_ROOT}")

Project root: D:\lh222k\AI-Research-Assistant-with-RAG


## 1. Chuẩn bị: index tài liệu mẫu vào vector store (in-memory)

Tái sử dụng hai tài liệu mẫu từ [`01_document_loading.ipynb`](../data_engineer/01_document_loading.ipynb) (tạo lại nếu chưa có, để notebook chạy độc lập — Yêu cầu 9.1).

In [2]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

sample_files = {
    "sample_rag_overview.txt": (
        "Retrieval-Augmented Generation (RAG) la kien truc ket hop retrieval va "
        "generation. He thong tim cac doan van ban lien quan tu kho du lieu rieng "
        "truoc khi yeu cau LLM sinh cau tra loi, giup giam hien tuong ao giac va "
        "bam sat nguon tai lieu thuc te."
    ),
    "sample_chunking_notes.md": (
        "# Ghi chu ve Text Chunking\n\nChia nho van ban (chunking) la buoc quan "
        "trong truoc khi tao embedding. Chunk qua lon lam loang ngu canh, chunk qua "
        "nho lam mat ngu canh xung quanh. Tham so chunk_overlap giup giu lien ket "
        "giua cac chunk lien tiep, tranh cat dut y giua cau."
    ),
}
for name, text in sample_files.items():
    path = RAW_DIR / name
    if not path.exists():
        path.write_text(text, encoding="utf-8")
        print(f"Da tao: {path.relative_to(PROJECT_ROOT)}")
    else:
        print(f"Da ton tai: {path.relative_to(PROJECT_ROOT)}")

loader = DocumentLoader()
chunker = TextChunker(strategy=ChunkStrategy.RECURSIVE, chunk_size=220, chunk_overlap=40)
embedder = OllamaEmbeddingModel(model_name="nomic-embed-text")
store = ChromaVectorStore(collection_name="notebook_retrieval_strategies", persist_dir=None)

all_chunks = []
for name in sample_files:
    document = loader.load(str(RAW_DIR / name))
    chunks = chunker.chunk(document)
    vectors = embedder.embed_batch([c.content for c in chunks])
    store.add(chunks, vectors)
    all_chunks.extend(chunks)

print(f"\nDa index {len(all_chunks)} chunk tu {len(sample_files)} tai lieu vao '{store.collection_name}'.")
print(f"Thong ke: {store.get_collection_stats()}")

Da ton tai: data\raw\sample_rag_overview.txt
Da ton tai: data\raw\sample_chunking_notes.md



Da index 12 chunk tu 2 tai lieu vao 'notebook_retrieval_strategies'.
Thong ke: {'collection_name': 'notebook_retrieval_strategies', 'num_chunks': 12, 'persist_dir': None, 'in_memory': True}


## 2. Dense retrieval — `Retriever` bọc `similarity_search` (so khớp theo ý nghĩa)

Câu hỏi được embed thành vector rồi so khớp với vector của các chunk đã lưu — đây chính là luồng `RAGPipeline.query()` dùng (sequence diagram design.md §1.3: `RT->>VS: similarity_search(query_vector, k)`).

In [3]:
retriever = Retriever(vector_store=store, top_k=3)

QUESTION = "He thong RAG hoat dong nhu the nao de tra loi cau hoi?"
query_vector = embedder.embed_text(QUESTION)

dense_results = retriever.retrieve(query_vector)
print(f"Cau hoi: {QUESTION!r}\n")
print("Dense (embedding-based):")
for sc in dense_results:
    preview = sc.chunk.content.strip().replace("\n", " ")[:90]
    print(f"  #{sc.rank} score={sc.score:.3f}  {preview!r}...")

Cau hoi: 'He thong RAG hoat dong nhu the nao de tra loi cau hoi?'

Dense (embedding-based):
  #1 score=0.906  'thong RAG se tim kiem cac doan van ban lien quan tu kho du lieu rieng truoc khi yeu cau LL'...
  #2 score=0.877  'cau tra loi). Du an nay trien khai toan bo quy trinh do bang cac thanh phan chay hoan toan'...
  #3 score=0.877  'tao ra phan chong lap giua cac doan lien tiep de khong mat ngu canh o ranh gioi chunk - da'...


## 3. Sparse retrieval — so khớp theo từ khoá (baseline kiểu TF-IDF/BM25)

Cài đặt rất đơn giản chỉ để **minh hoạ** — đếm số từ (token) chung giữa câu hỏi và nội dung chunk, chuẩn hoá theo độ dài hợp (Jaccard similarity). Không thay thế cho BM25 thật, nhưng đủ để thấy đặc điểm "nhạy với từ khoá trùng khớp chính xác" của truy xuất sparse.

In [4]:
_TOKEN_RE = re.compile(r"[\w\u00c0-\u1ec9]+", re.UNICODE)


def tokenize(text):
    return set(tok.lower() for tok in _TOKEN_RE.findall(text))


def sparse_score(query, content):
    q_tokens, c_tokens = tokenize(query), tokenize(content)
    if not q_tokens or not c_tokens:
        return 0.0
    overlap = q_tokens & c_tokens
    union = q_tokens | c_tokens
    return len(overlap) / len(union)  # Jaccard in [0, 1]


sparse_scored = sorted(
    ((sparse_score(QUESTION, c.content), c) for c in all_chunks),
    key=lambda pair: pair[0],
    reverse=True,
)[:3]

print("Sparse (keyword overlap / Jaccard):")
for rank, (score, chunk) in enumerate(sparse_scored, start=1):
    preview = chunk.content.strip().replace("\n", " ")[:90]
    print(f"  #{rank} score={score:.3f}  {preview!r}...")

Sparse (keyword overlap / Jaccard):
  #1 score=0.122  'ket noi voi OLLAMA cho ca embedding lan sinh van ban. Su phoi hop giua ba vai tro nay - da'...
  #2 score=0.104  'thong RAG se tim kiem cac doan van ban lien quan tu kho du lieu rieng truoc khi yeu cau LL'...
  #3 score=0.080  'cau tra loi). Du an nay trien khai toan bo quy trinh do bang cac thanh phan chay hoan toan'...


## 4. Hybrid retrieval — kết hợp dense + sparse

Trộn điểm dense và sparse theo trọng số (`alpha`) rồi xếp hạng lại trên **hợp** các chunk xuất hiện ở một trong hai danh sách — một cách tiếp cận hybrid phổ biến khi muốn vừa "hiểu ý nghĩa" vừa "bắt đúng từ khoá".

In [5]:
ALPHA = 0.6  # trong so cho diem dense; (1 - ALPHA) cho diem sparse

dense_by_id = {sc.chunk.chunk_id: sc.score for sc in dense_results}
sparse_by_id = {chunk.chunk_id: score for score, chunk in sparse_scored}
chunk_by_id = {c.chunk_id: c for c in all_chunks}

candidate_ids = set(dense_by_id) | set(sparse_by_id)
hybrid = [
    (ALPHA * dense_by_id.get(cid, 0.0) + (1 - ALPHA) * sparse_by_id.get(cid, 0.0), chunk_by_id[cid])
    for cid in candidate_ids
]
hybrid.sort(key=lambda pair: pair[0], reverse=True)

print(f"Hybrid (alpha={ALPHA} dense + {1 - ALPHA:.1f} sparse):")
for rank, (score, chunk) in enumerate(hybrid[:3], start=1):
    preview = chunk.content.strip().replace("\n", " ")[:90]
    print(f"  #{rank} score={score:.3f}  {preview!r}...")

Hybrid (alpha=0.6 dense + 0.4 sparse):
  #1 score=0.585  'thong RAG se tim kiem cac doan van ban lien quan tu kho du lieu rieng truoc khi yeu cau LL'...
  #2 score=0.558  'cau tra loi). Du an nay trien khai toan bo quy trinh do bang cac thanh phan chay hoan toan'...
  #3 score=0.526  'tao ra phan chong lap giua cac doan lien tiep de khong mat ngu canh o ranh gioi chunk - da'...


## 5. So sánh nhanh ba chiến lược

Bảng dưới liệt kê `chunk_id` đứng đầu mỗi chiến lược — quan sát xem chúng có đồng thuận hay không. Nếu câu hỏi dùng từ ngữ khác với tài liệu (diễn giải lại ý), dense thường "thắng thế"; nếu câu hỏi lặp lại đúng thuật ngữ trong tài liệu, sparse có thể bắt kịp hoặc vượt trội.

In [6]:
top1 = {
    "Dense": dense_results[0].chunk.chunk_id if dense_results else None,
    "Sparse": sparse_scored[0][1].chunk_id if sparse_scored else None,
    "Hybrid": hybrid[0][1].chunk_id if hybrid else None,
}
for strategy, chunk_id in top1.items():
    print(f"  Top-1 theo {strategy:7s}: {chunk_id}")

agree = len(set(top1.values())) == 1
print(f"\nCa ba chien luoc dong thuan ve ket qua #1: {agree}")

  Top-1 theo Dense  : 4c69f82b6454088f_chunk_0001
  Top-1 theo Sparse : 4c69f82b6454088f_chunk_0005
  Top-1 theo Hybrid : 4c69f82b6454088f_chunk_0001

Ca ba chien luoc dong thuan ve ket qua #1: False


## 6. Xác minh Property 6, 7, 8 cho dense retrieval

`similarity_search()` (qua `Retriever`) phải đảm bảo: `len(result) <= k`, sắp xếp **giảm dần** theo `score`, và mỗi `score ∈ [0.0, 1.0]` (design.md Phần 3, Property 6-8 — Validates Yêu cầu 4.2-4.4).

In [7]:
k = retriever.top_k
assert len(dense_results) <= k, "Property 6: khong vuot qua top_k"
scores = [sc.score for sc in dense_results]
assert scores == sorted(scores, reverse=True), "Property 7: phai sap xep giam dan theo score"
assert all(0.0 <= s <= 1.0 for s in scores), "Property 8: score phai thuoc [0.0, 1.0]"
print(f"Property 6 OK: len(ket qua)={len(dense_results)} <= k={k}")
print(f"Property 7 OK: scores giam dan -> {[round(s, 3) for s in scores]}")
print(f"Property 8 OK: moi score thuoc [0.0, 1.0]")

Property 6 OK: len(ket qua)=3 <= k=3
Property 7 OK: scores giam dan -> [0.906, 0.877, 0.877]
Property 8 OK: moi score thuoc [0.0, 1.0]


## 7. Tổng kết

- **Dense** (qua `Retriever`/`ChromaVectorStore.similarity_search`) so khớp theo *ý nghĩa* nhờ embedding — mạnh khi câu hỏi diễn đạt khác từ ngữ so với tài liệu gốc.
- **Sparse** (baseline Jaccard/keyword-overlap minh hoạ trong notebook) nhạy với từ khoá trùng khớp chính xác — đơn giản, nhanh, nhưng "cứng nhắc" về mặt ngôn ngữ.
- **Hybrid** kết hợp cả hai — một hướng thực nghiệm hữu ích khi muốn cân bằng giữa "hiểu ý nghĩa" và "bắt đúng từ khoá", đặc biệt với các câu hỏi chứa thuật ngữ chuyên ngành.
- **Property 6/7/8** được xác minh trực tiếp cho kết quả `similarity_search` — nền tảng đảm bảo `RAGPipeline.query()` luôn nhận được context hợp lệ để xây prompt.